# Forecast Visualization and Economic Context Dashboard

Interactive notebook for communicating forecast quality and macro-financial narratives.

Features:
- Actual vs forecast overlays for key models
- Cumulative absolute-error comparison
- Volatility-period zooming
- Economic event annotation template

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')
from statistical_validation import load_config, discover_forecasts, event_annotation_frame

Active target: PHP
Rows: 6345


#### Interpretation
This cell loads data for visualization and confirms active modeling scope. Verify sample coverage before interpreting any chart-level conclusion.

In [8]:
for pair in sorted(plot_df['Pair'].unique()):
    sub = plot_df[plot_df['Pair'] == pair].sort_values('Date')
    fig = go.Figure()
    actual = sub[sub['Model'] == top_models[0]][['Date', 'Actual']].drop_duplicates()
    fig.add_trace(go.Scatter(x=actual['Date'], y=actual['Actual'], mode='lines', name='Actual', line=dict(color='black', width=2)))
    for m in top_models:
        msub = sub[sub['Model'] == m]
        fig.add_trace(go.Scatter(x=msub['Date'], y=msub['Forecast'], mode='lines', name=m))
    fig.update_layout(template='plotly_white', title=f'{pair}: forecast dashboard', width=1150, height=450)
    fig.show()

#### Interpretation
The first plot compares forecast levels against realized series. Better models track turning points with smaller lag and reduced overshooting.

In [9]:
err = plot_df.copy()
err['abs_error'] = err['Error'].abs()
err['cum_abs_error'] = err.sort_values('Date').groupby(['Pair', 'Model'])['abs_error'].cumsum()

events = event_annotation_frame()
events['Date'] = pd.to_datetime(events['Date'])

for pair in sorted(err['Pair'].unique()):
    sub = err[err['Pair'] == pair]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for m in top_models:
        msub = sub[sub['Model'] == m].sort_values('Date')
        fig.add_trace(go.Scatter(x=msub['Date'], y=msub['Forecast'], mode='lines', name=f'{m} forecast'), secondary_y=False)
    actual = sub[sub['Model'] == top_models[0]][['Date', 'Actual']].drop_duplicates().sort_values('Date')
    fig.add_trace(go.Scatter(x=actual['Date'], y=actual['Actual'], mode='lines', name='Actual', line=dict(color='black', width=2)), secondary_y=False)
    vol_proxy = sub.groupby('Date', as_index=False)['abs_error'].mean().sort_values('Date')
    fig.add_trace(go.Scatter(x=vol_proxy['Date'], y=vol_proxy['abs_error'], mode='lines', name='Realized abs error', line=dict(dash='dot', color='firebrick')), secondary_y=True)
    for _, ev in events.iterrows():
        fig.add_vline(x=ev['Date'], line_dash='dot', line_color='gray', opacity=0.35)
        fig.add_annotation(x=ev['Date'], y=1.02, yref='paper', text=ev['Event'], showarrow=False, font=dict(size=9), textangle=-90)
    fig.update_layout(
        template='plotly_white',
        title=f'{pair}: actual vs forecast with event markers',
        width=1300,
        height=550,
        xaxis=dict(rangeslider=dict(visible=True)),
        legend=dict(orientation='h')
    )
    fig.update_yaxes(title_text='Log-return (%)', secondary_y=False)
    fig.update_yaxes(title_text='Realized abs error', secondary_y=True)
    fig.show()

#### Interpretation
This panel visualizes forecast errors through time and by model. Systematic bias or clustered large errors can indicate regime-dependent misspecification.

## Economic Event Overlay Template

Add a CSV with columns `Date`, `Event`, `Category` (policy, risk, trade) and merge with forecast-error spikes.

Suggested events to annotate for PHP analysis:
- BSP policy-rate surprises
- Federal Reserve policy shifts
- High-VIX global risk-off episodes
- Major China growth/trade surprises
- Balance-of-payments and remittance shocks

In [7]:
out_dir = f'results/{active_target}/evaluation'
os.makedirs(out_dir, exist_ok=True)

if 'err' not in globals():
    err = plot_df.copy()
    err['abs_error'] = err['Error'].abs()
    err['cum_abs_error'] = err.sort_values('Date').groupby(['Pair', 'Model'])['abs_error'].cumsum()

err.to_csv(f'{out_dir}/forecast_dashboard_error_paths.csv', index=False)
print('Saved:', f'{out_dir}/forecast_dashboard_error_paths.csv')

Saved: results/PHP/evaluation/forecast_dashboard_error_paths.csv


#### Interpretation
The saved figure confirmation indicates all visualization artifacts were exported. These files should be used as canonical plots in the final narrative.

### Narrative for the dashboard

Interpret the event markers as candidate stress periods. If a hybrid model flattens its error line faster after a Fed or VIX shock than the linear VAR, that is evidence it is capturing nonlinear adjustment dynamics in PHP FX rather than merely fitting in-sample noise.